In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import decoupler as dc
import matplotlib.pyplot as plt
import scipy.io
import scipy.sparse as sp

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "03_fibroblast_vs_SF"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_human_annotated.zarr")
fib = adata[adata.obs["cell_largetype"] == "Fibroblast"].copy()
fib.obs["roi_name"] = fib.obs["roi_name"].astype(str)
fib.obs["group"] = fib.obs["roi_name"].replace({"SF1": "SF", "SF2": "SF"})
fib.X = fib.layers["log1p_norm"]
fib.obs.groupby("roi_name")["total_counts"].agg(["size", "median"])

In [ ]:
def deg(a, b):
    a_mask = (fib.obs["group" if a == "SF" else "roi_name"] == a).values
    b_mask = (fib.obs["group" if b == "SF" else "roi_name"] == b).values
    sub = fib[a_mask | b_mask].copy()
    sub.obs["g"] = np.where(a_mask[a_mask | b_mask], a, b)
    sc.tl.rank_genes_groups(
        sub,
        "g",
        groups=[a],
        reference=b,
        method="wilcoxon",
        pts=True,
    )
    results_df = sc.get.rank_genes_groups_df(sub, group=a).rename(columns={"names": "gene", "scores": "stat", "logfoldchanges": "log2FoldChange", "pvals": "pvalue", "pvals_adj": "padj"}).set_index("gene")
    pts = sub.uns["rank_genes_groups"]["pts"]
    results_df["pct_a"], results_df["pct_b"] = pts[a], pts[b]
    detected = results_df[["pct_a", "pct_b"]].max(axis=1) >= 0.1
    results_df["sig"] = (results_df["padj"] < 0.05) & (results_df["log2FoldChange"].abs() > 0.5) & detected
    return results_df

pairs = [("Xray_Late", "SF1"), ("Xray_Late", "SF2"), ("Xray_Late", "SF"),
         ("Cion_Late", "SF1"), ("Cion_Late", "SF2"), ("Cion_Late", "SF"),
         ("Xray_Under", "SF1"), ("Xray_Under", "SF2"), ("Xray_Under", "SF"),
         ("Xray_Under", "Xray_Late"), ("Xray_Under", "Cion_Late")]

results = {}
for a, b in pairs:
    results[(a, b)] = deg(a, b)
    results[(a, b)].to_csv(OUT_DIR / f"deg_{a}_vs_{b}.csv")

pd.DataFrame({f"{a} vs {b}": {"up in A": int((r["sig"] & (r["log2FoldChange"] > 0)).sum()),
                              "up in B": int((r["sig"] & (r["log2FoldChange"] < 0)).sum())}
              for (a, b), r in results.items()}).T

In [ ]:
def both(x, direction):
    r1, r2 = results[(x, "SF1")], results[(x, "SF2")]
    g1 = r1.index[r1["sig"] & (np.sign(r1["log2FoldChange"]) == direction)]
    g2 = r2.index[r2["sig"] & (np.sign(r2["log2FoldChange"]) == direction)]
    return sorted(set(g1) & set(g2))

for x in ["Xray_Under", "Xray_Late", "Cion_Late"]:
    print(x, "> SF1 and SF2:", both(x, 1))
    print(x, "< SF1 and SF2:", both(x, -1))

In [ ]:
def volcano(results_df, xlabel, path, top_genes):
    fig, ax = plt.subplots(figsize=(fs.COL1, 70 * fs.MM), layout="constrained")
    dc.pl.volcano(results_df.dropna(subset=["padj"]), x="log2FoldChange", y="padj",
                  top=top_genes,
                  color_pos=fs.UP, color_neg=fs.DOWN, color_null=fs.LIGHTGREY,
                  kw_scatter={"s": 4, "lw": 0, "rasterized": True}, ax=ax)
    for line in ax.lines:
        line.set(color=fs.GREY, lw=0.4, ls=(0, (2, 2)))
    for text in ax.texts:
        text.set(fontsize=5, fontstyle="italic")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("−log$_{10}$(adjusted $\\it{P}$ value)")
    fs.save(fig, path)
    plt.show()

for a, b in [("Xray_Late", "SF"), ("Cion_Late", "SF"), ("Xray_Under", "SF"), ("Xray_Under", "Xray_Late")]:
    results_df = results[(a, b)]
    volcano(results_df[results_df[["pct_a", "pct_b"]].max(axis=1) >= 0.1], f"log$_2$ fold change ({fs.label(a)} / {fs.label(b)})",
            OUT_DIR / f"volcano_{a}_vs_{b}", [])

In [ ]:
genes = ["COL1A1", "COL1A2", "COL3A1", "COL5A1", "COL5A2", "FN1", "POSTN", "ELN", "FBLN5", "MFAP5",
         "SERPINE1", "SNAI2", "CCN2", "CTHRC1", "LOX", "TNC", "THBS1", "TGFB1", "TIMP1", "SPARC"]
genes = [g for g in genes if g in fib.var_names]

lfc = pd.DataFrame({f"{a} vs {b}": r.loc[genes, "log2FoldChange"].round(2).astype(str)
                    + np.where(r.loc[genes, "sig"], " *", "") for (a, b), r in results.items()})
lfc.to_csv(OUT_DIR / "fibrosis_genes_log2fc.csv")
lfc

In [ ]:
d = BASE / "data" / "fibroblast_counts"
d.mkdir(exist_ok=True)
counts = sp.csr_matrix(fib.layers["counts"]).astype(np.int64)
with open(d / "matrix.mtx", "wb") as f:
    scipy.io.mmwrite(f, counts.T.tocoo())
pd.Series(fib.var_names).to_csv(d / "genes.tsv", index=False, header=False)
fib.obs[["roi_name", "group"]].to_csv(d / "cells.tsv", sep="\t")